# Flood mapping pipeline on Colab

Runs the whole workflow from the repository zip on Google Drive: install, data, smoke test, baseline model, continual learning.

**Every session (also after a disconnect):** run cells 1–4 in order, then the step you need. Every step is safe to re-run: finished work is detected and skipped, and interrupted training continues from its last checkpoint.

**Code zip:** upload the repository zip to the runtime (Files pane → `/content`) at the start of each session; a new runtime starts empty. If it isn't there, cell 2 also looks for the same file name in `DRIVE_ROOT`. If the zip has a different name, change `REPO_ZIP` in cell 1.

**Data:** training reads the tiles from the runtime's local disk (`/content/data`), restored from the zip on Drive, because reading tiles straight from Drive is far slower. Checkpoints, logs and results are written to Drive so they survive a disconnect.

Use a GPU runtime (Runtime → Change runtime type → GPU).

## 1. Settings
Edit these once. Everything below reads them.

In [ ]:
DRIVE_ROOT    = "/content/drive/MyDrive/floodmap"
REPO_ZIP      = "/content/flood_extent_mapping.zip"   # uploaded to the runtime (falls back to DRIVE_ROOT)
PROCESSED_ZIP = f"{DRIVE_ROOT}/processed_512.zip"    # processed tiles; created by step 3 if it doesn't exist
RAW_ZIP       = "/content/drive/MyDrive/mmflood.zip"  # raw MMFlood zip; only needed when PROCESSED_ZIP doesn't exist yet
ARTIFACTS_DIR = f"{DRIVE_ROOT}/runs"                  # all runs, logs and results (on Drive, so they survive disconnects)

THRESHOLD  = "0.80"    # operating threshold (out-of-fold event-CV calibration)
MODALITIES = "vv vh"

# Local working folders (fast runtime disk; rebuilt each session)
REPO_DIR      = "/content/floodmap_repo"
RAW_DIR       = "/content/data/raw"
PROCESSED_DIR = "/content/data/processed_512"

import os
os.environ.update(DRIVE_ROOT=DRIVE_ROOT, ARTIFACTS_DIR=ARTIFACTS_DIR, REPO_DIR=REPO_DIR, RAW_DIR=RAW_DIR,
                  PROCESSED_DIR=PROCESSED_DIR, SUMMARY_FILE=f"{PROCESSED_DIR}/activations.json",
                  THRESHOLD=THRESHOLD, MODALITIES=MODALITIES)

## 2. Mount Drive and install the code
A clean install from `REPO_ZIP` every session (2–4 minutes).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, subprocess, sys, zipfile
from pathlib import Path

if not Path(REPO_ZIP).exists() and (Path(DRIVE_ROOT) / Path(REPO_ZIP).name).exists():
    REPO_ZIP = str(Path(DRIVE_ROOT) / Path(REPO_ZIP).name)
if not Path(REPO_ZIP).exists():
    zips = sorted(str(p) for p in Path("/content").glob("*.zip")) + sorted(str(p) for p in Path(DRIVE_ROOT).glob("*.zip"))
    raise FileNotFoundError(f"{REPO_ZIP} not found. Upload it to /content (Files pane), or set REPO_ZIP in cell 1. Zips found: {zips}")
print("Installing from", REPO_ZIP)

os.chdir("/content")
shutil.rmtree(REPO_DIR, ignore_errors=True)
shutil.rmtree("/content/_repo_unzip", ignore_errors=True)
with zipfile.ZipFile(REPO_ZIP) as z:
    top = z.namelist()[0].split("/")[0]
    z.extractall("/content/_repo_unzip")
shutil.move(f"/content/_repo_unzip/{top}", REPO_DIR)
shutil.rmtree("/content/_repo_unzip", ignore_errors=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import importlib, floods.colab
importlib.reload(floods.colab)
from floods.colab import run, unzip, zip_dir, find_file, best_checkpoint

run(f'cd "{REPO_DIR}" && FLOODMAP_SKIP_TESTS=1 bash scripts/install_colab.sh > /content/install_output.txt 2>&1 '
    f'|| (tail -n 40 /content/install_output.txt; exit 1)')
run("tail -n 6 /content/install_output.txt")
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
run("python -c 'import floods, torch; print(\"floods\", floods.__version__, \"| torch\", torch.__version__, \"| GPU:\", torch.cuda.is_available())'")

## 3. Data
Uses the processed tiles if they're already on this runtime, otherwise restores them from `PROCESSED_ZIP`. Only if that zip doesn't exist does it unzip the raw data, preprocess (about an hour) and save `PROCESSED_ZIP` to Drive for next time.

In [ ]:
processed = Path(PROCESSED_DIR)

def _flatten(root: Path):
    """If the zip had an extra top folder, move its contents up so root/train exists."""
    if (root / "train").exists():
        return
    for sub in root.iterdir():
        if sub.is_dir() and (sub / "train").exists():
            for item in sub.iterdir():
                shutil.move(str(item), root / item.name)
            sub.rmdir()
            return

if (processed / "train").exists():
    print("Processed tiles already on this runtime:", processed)
elif Path(PROCESSED_ZIP).exists():
    unzip(PROCESSED_ZIP, processed, desc="restore processed tiles")
    _flatten(processed)
else:
    others = [str(z) for z in Path(DRIVE_ROOT).rglob("*.zip") if "process" in z.name.lower()]
    if others:
        raise FileNotFoundError(f"{PROCESSED_ZIP} not found, but these look like processed-data zips: {others}. "
                                "Set PROCESSED_ZIP in cell 1 to the right one (or delete them to rebuild).")
    if not Path(RAW_ZIP).exists():
        zips = [str(z) for z in Path("/content/drive/MyDrive").glob("*.zip")] + [str(z) for z in Path(DRIVE_ROOT).glob("*.zip")]
        raise FileNotFoundError(f"Neither PROCESSED_ZIP nor RAW_ZIP exists. Zips on Drive: {zips}. Fix the paths in cell 1.")
    print("No processed zip yet: building it from the raw data (about an hour).")
    unzip(RAW_ZIP, RAW_DIR, desc="unzip raw MMFlood")
    raw_summary = find_file(RAW_DIR, "activations.json")
    if raw_summary is None:
        raise FileNotFoundError(f"activations.json not found inside {RAW_ZIP}")
    raw_root = raw_summary.parent
    run(f"""
cd "$REPO_DIR"
floodmap preprocess \\
  --config configs/reproduction/preprocess_512.yaml \\
  --raw-data-dir "{raw_root}" \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --summary-file "{raw_summary}" \\
  --subset train val test \\
  --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "$ARTIFACTS_DIR/preprocess.log"
""")
    shutil.copyfile(raw_summary, processed / "activations.json")
    zip_dir(processed, PROCESSED_ZIP, desc="zip processed tiles")

if not (processed / "activations.json").exists():
    found = find_file(processed, "activations.json")
    if found is None:
        raise FileNotFoundError("activations.json is missing from the processed data; it is needed for continual learning.")
    shutil.copyfile(found, processed / "activations.json")

for split in ("train", "val", "test"):
    n = len(list((processed / split).rglob("*.tif")))
    print(f"{split:5s}: {n} files")

## 4. Smoke test
A few minutes on CPU. Runs every command (preprocess → train → evaluate → continual learning + resume) on tiny synthetic data. Run it after every code update; if anything says FAIL, stop and fix that first.

In [ ]:
run('cd "$REPO_DIR" && python scripts/pipeline_smoke_test.py --clean')

## 5. Reference model

Event-CV → threshold calibration → final training → one test evaluation → deployment bundle. These steps take several hours and are switched off by default; set `RUN_BASELINE = True` to run them. Each step skips work that is already finished.

Reference results: event-CV fold 0 event-macro F1 0.478 (VV+VH); out-of-fold threshold 0.80 (event-macro F1 0.424); test F1 0.520 at 0.80.

In [ ]:
RUN_BASELINE = False

### 5a. Event-level cross-validation (folds 0 and 1, ~3 h)

In [ ]:
if RUN_BASELINE:
    run("""
cd "$REPO_DIR"
floodmap event-cv \\
  --config configs/reproduction/final_unet_resnet34_vv_vh.yaml \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --output-dir "$ARTIFACTS_DIR/event_cv" \\
  --candidates imagenet:unet:resnet34:reference_unet \\
  --modality-sets vv+vh \\
  --folds 5 --fold-indices 0 1 \\
  --epochs 12 --patience 3 \\
  --batch-size 8 --num-workers 4 \\
  --pretrained --amp --gpu \\
  --thresholds 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45 0.50 0.55 0.60 0.65 0.70 0.75 0.80 \\
  --skip-completed \\
  --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "$ARTIFACTS_DIR/event_cv/output.log"
""")

### 5b. Threshold calibration on out-of-fold predictions

In [ ]:
if RUN_BASELINE:
    run("""
cd "$REPO_DIR"
floodmap calibrate-event-cv \\
  --cv-dir "$ARTIFACTS_DIR/event_cv" \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --output-dir "$ARTIFACTS_DIR/event_cv_calibration" \\
  --candidate reference_unet \\
  --modalities vv+vh \\
  --fold-indices 0 1 \\
  --thresholds 0.50 0.55 0.60 0.65 0.70 0.75 0.80 0.85 0.90 0.95 \\
  --amp --gpu \\
  --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "$ARTIFACTS_DIR/event_cv_calibration/output.log"
""")

### 5c. Final model (~25 min)
Resumes automatically if a previous attempt was interrupted.

In [ ]:
FINAL_RUN = "final_unet_r34"
if RUN_BASELINE:
    resume = "--resume" if (Path(ARTIFACTS_DIR) / FINAL_RUN / "models" / "last.ckpt").exists() else ""
    run(f"""
cd "$REPO_DIR"
floodmap train \\
  --config configs/reproduction/final_unet_resnet34_vv_vh.yaml \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --output-folder "$ARTIFACTS_DIR" \\
  --run-id {FINAL_RUN} \\
  --encoder resnet34 --decoder unet \\
  --input-modalities $MODALITIES \\
  --epochs 6 --patience 99 \\
  --batch-size 8 --num-workers 4 \\
  --seed 42 --pretrained --amp --gpu \\
  --augmentation-profile composite \\
  --threshold-sweep --thresholds $THRESHOLD --threshold-metric f1 --monitor-threshold-sweep \\
  --save-last \\
  --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "$ARTIFACTS_DIR/{FINAL_RUN}/output.log" {resume}
""")

### 5d. Test evaluation (once) and deployment bundle

In [ ]:
if RUN_BASELINE:
    run_dir = Path(ARTIFACTS_DIR) / FINAL_RUN
    CKPT = best_checkpoint(run_dir)
    print("Best checkpoint:", CKPT)
    run(f"""
cd "$REPO_DIR"
floodmap evaluate \\
  --config "{run_dir}/config.yaml" \\
  --checkpoint "{CKPT}" \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --split test \\
  --batch-size 1 --num-workers 2 \\
  --input-modalities $MODALITIES \\
  --amp --gpu \\
  --thresholds $THRESHOLD --threshold-metric f1 --metric-mode global \\
  --inference-mode sliding_window --window-size 512 --window-overlap 128 \\
  --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "{run_dir}/test_evaluation.log"

floodmap export-deployment \\
  --output-file "$ARTIFACTS_DIR/deployment/unet_resnet34_vv_vh/deployment_manifest.yaml" \\
  --model-name unet_resnet34_vv_vh_db \\
  --configs "{run_dir}/config.yaml" \\
  --checkpoints "{CKPT}" \\
  --threshold $THRESHOLD \\
  --input-modalities $MODALITIES \\
  --sar-units db10 \\
  --inference-mode sliding_window --window-size 512 --window-overlap 128
""")

## 6. Continual learning

All CL runs in one list. Re-running the cell after a disconnect continues where it stopped: finished strategies are only re-checked (seconds each), and the interrupted one resumes from its last epoch. Comment out lines you don't want.

| run | tasks | what |
|---|---|---|
| `cl_r34` | by year (2014–17, 2018–19, 2020–21) | all strategies, seed 42 |
| `cl_region_r34` | Europe → rest of world | all strategies, seed 42, buffer 100 |
| `cl_region_r34_s1`, `_s2` | Europe → rest of world | finetune / hybrid / joint, seeds 1 and 2 |
| `cl_region_r34_buf50`, `_buf150` | Europe → rest of world | hybrid with buffer 50 and 150 |

Asia alone is too small to be a task (54 training tiles, 1 test event), so it is merged into "rest of world". Check task sizes any time with the dry run.

### 6a. Dry run: task sizes, no training (1 minute)

In [ ]:
run("""
cd "$REPO_DIR"
floodmap continual-train \\
  --config configs/reproduction/final_unet_resnet34_vv_vh.yaml \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --output-folder "$ARTIFACTS_DIR" \\
  --run-id cl_dry_run \\
  --activations-json-path "$SUMMARY_FILE" \\
  --task-split region --task-regions europe asia+americas+africa+oceania \\
  --cl-threshold $THRESHOLD \\
  --input-modalities $MODALITIES \\
  --cl-eval-split test \\
  --cl-dry-run
""")

### 6b. CL runs

In [ ]:
YEAR_TASKS   = "--task-split year --task-year-ranges 2014-2017 2018-2019 2020-2021"
REGION_TASKS = "--task-split region --task-regions europe asia+americas+africa+oceania"
ALL_STRATEGIES = "finetune random entropy hybrid joint"

CL_RUNS = [
    # (run id,               tasks,        extra options)
    ("cl_r34",               YEAR_TASKS,   f"--strategies {ALL_STRATEGIES} --replay-buffer-size 100"),
    ("cl_region_r34",        REGION_TASKS, f"--strategies {ALL_STRATEGIES} --replay-buffer-size 100"),
    ("cl_region_r34_s1",     REGION_TASKS, "--seed 1 --strategies finetune hybrid joint --replay-buffer-size 100"),
    ("cl_region_r34_s2",     REGION_TASKS, "--seed 2 --strategies finetune hybrid joint --replay-buffer-size 100"),
    ("cl_region_r34_buf50",  REGION_TASKS, "--strategies hybrid --replay-buffer-size 50"),
    ("cl_region_r34_buf150", REGION_TASKS, "--strategies hybrid --replay-buffer-size 150"),
]

for run_id, tasks, extra in CL_RUNS:
    print(f"\n{'=' * 30} {run_id} {'=' * 30}", flush=True)
    run(f"""
cd "$REPO_DIR"
floodmap continual-train \\
  --config configs/reproduction/final_unet_resnet34_vv_vh.yaml \\
  --processed-data-dir "$PROCESSED_DIR" \\
  --output-folder "$ARTIFACTS_DIR" \\
  --run-id {run_id} \\
  --activations-json-path "$SUMMARY_FILE" \\
  {tasks} \\
  {extra} \\
  --epochs-per-task 5 --replay-batch-size 8 \\
  --cl-threshold $THRESHOLD \\
  --architecture unet --backbone resnet34 --pretrained \\
  --image-size 512 --batch-size 8 --num-workers 4 \\
  --input-modalities $MODALITIES \\
  --augmentation-profile composite \\
  --threshold-sweep --threshold-metric f1 \\
  --cl-eval-split test --cl-eval-inference-mode sliding_window \\
  --window-size 512 --window-overlap 128 \\
  --gpu --amp --progress --dynamic-progress --heartbeat-seconds 60 \\
  --log-file "$ARTIFACTS_DIR/{run_id}.log" \\
  --resume
""")

### 6c. Results
Mean ± std over seeds, and the buffer-size comparison. Missing runs are listed as warnings. Writes `cl_region_summary.csv` next to the runs.

In [ ]:
run("""
cd "$REPO_DIR"
echo "Year split (seed 42):"
python scripts/aggregate_cl_runs.py --runs-dir "$ARTIFACTS_DIR" --run year=cl_r34
echo
echo "Region split:"
python scripts/aggregate_cl_runs.py --runs-dir "$ARTIFACTS_DIR" \\
  --run seeds=cl_region_r34 --run seeds=cl_region_r34_s1 --run seeds=cl_region_r34_s2 \\
  --run buffer50=cl_region_r34_buf50 --run buffer100=cl_region_r34 --run buffer150=cl_region_r34_buf150 \\
  --output "$ARTIFACTS_DIR/cl_region_summary.csv"
""")

## 7. Streamlit app

7a exports the trained model into the app's model catalogue; 7b starts the app here in Colab and prints a link to open it. Needs cells 1–2 only (not the data cells).

### 7a. Export the deployment bundle into the app (seconds)

In [ ]:
FINAL_RUN = "final_unet_r34"
BUNDLE_NAME = "unet_resnet34_vv_vh"
run_dir = Path(ARTIFACTS_DIR) / FINAL_RUN
bundle_on_drive = Path(ARTIFACTS_DIR) / "deployment" / BUNDLE_NAME
CKPT = best_checkpoint(run_dir)
if CKPT is None:
    raise FileNotFoundError(f"No best checkpoint in {run_dir}/models; train the final model first (5c).")
print("Model checkpoint:", CKPT)

# Export (or re-export) a portable bundle to Drive, then copy it into the app's model catalogue.
shutil.rmtree(bundle_on_drive, ignore_errors=True)
run(f"""
cd "$REPO_DIR"
floodmap export-deployment \\
  --output-file "{bundle_on_drive}/deployment_manifest.yaml" \\
  --model-name unet_resnet34_vv_vh_db \\
  --configs "{run_dir}/config.yaml" \\
  --checkpoints "{CKPT}" \\
  --threshold $THRESHOLD \\
  --input-modalities $MODALITIES \\
  --sar-units db10 \\
  --inference-mode sliding_window --window-size 512 --window-overlap 128 \\
  --notes "U-Net ResNet34, Sentinel-1 VV+VH in dB; threshold from out-of-fold event-CV."
""")
app_bundle = Path(REPO_DIR) / "streamlit_app" / "deployments" / BUNDLE_NAME
shutil.rmtree(app_bundle, ignore_errors=True)
shutil.copytree(bundle_on_drive, app_bundle)
run(f'grep -E "model_name|sar_units|normalization_mode|threshold" "{app_bundle}/deployment_manifest.yaml"')
print("App bundle:", app_bundle)

### 7b. Run the app in Colab

Starts Streamlit in the background and opens a temporary public link through a Cloudflare quick tunnel. Anyone with the link can use the app while this runtime is alive, so don't share it widely. Re-run the cell to restart the app.

In [ ]:
import re, time, urllib.request
run('pip install -q "streamlit==1.62.0"')
if not Path("/content/cloudflared").exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

def stop_app():
    subprocess.run("pkill -f 'streamlit run'; pkill -f 'cloudflared tunnel'", shell=True)

stop_app()
app_log = open("/content/streamlit.log", "w")
subprocess.Popen(["streamlit", "run", f"{REPO_DIR}/streamlit_app/app.py", "--server.headless", "true",
                  "--server.port", "8501", "--server.maxUploadSize", "2000"],
                 cwd=REPO_DIR, stdout=app_log, stderr=subprocess.STDOUT)
tunnel_log = open("/content/cloudflared.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=tunnel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    found = re.findall(r"https://[a-z0-9-]+[.]trycloudflare[.]com", Path("/content/cloudflared.log").read_text())
    if found:
        url = found[-1]
        break
print("Open the app:", url or "tunnel not ready yet; see /content/cloudflared.log")
print("App log: /content/streamlit.log | to stop the app run: stop_app()")